## Differential Evolution - DE

### Ackley test function (for N dimensions)

In [7]:
import numpy as np

def ackley(x):
    n = len(x)
    sum1 = np.sum(x**2)
    sum2 = np.sum(np.cos(2 * np.pi * x))

    term1 = -20.0 * np.exp(-0.2 * np.sqrt(sum1 / n))
    term2 = -np.exp(sum2 / n)

    return term1 + term2 + 20 + np.e

### Main differential evolution algorithm

In [15]:
def differential_evolution_experiment(
    objective_func,
    bounds,
    pop_size=30,
    F=0.5,
    CR=0.7,
    max_iter=200,
    target_quality=0.01,
):
    """Parameters:

    objective_func: the objective function to be minimized
    bounds: list of tuples defining the boundaries for each dimension [(min,
    max), ...]
    pop_size: number of individuals in the population
    F: mutation factor (differential weight, typically 0.4 - 1.0)
    CR: crossover rate (probability of crossover, 0.0 - 1.0)
    max_iter: number of generations / iterations
    target_quality: threshold to measure the convergence speed
    """
    dimensions = len(bounds)

    # Initialize the population with random values within the specified bounds
    pop = np.zeros((pop_size, dimensions))
    for i in range(dimensions):
        pop[:, i] = np.random.uniform(bounds[i][0], bounds[i][1], pop_size)

    # Evaluate the initial population
    fitness = np.array([objective_func(ind) for ind in pop])

    # Find the best individual at the start
    best_idx = np.argmin(fitness)
    best_vector = pop[best_idx]
    best_score = fitness[best_idx]

    # Track when the algorithm reaches the desired quality
    generation_reached_target = None

    # Main evolutionary loop
    for generation in range(max_iter):
        # Check if target quality is reached for the first time
        if best_score <= target_quality and generation_reached_target is None:
            generation_reached_target = generation

        for i in range(pop_size):
            # Mutation - select 3 random individuals, distinct from the current 'i'
            candidates = [idx for idx in range(pop_size) if idx != i]
            r1, r2, r3 = np.random.choice(candidates, 3, replace=False)

            # Keep values within the allowable boundaries
            mutated = pop[r1] + F * (pop[r2] - pop[r3])

            for d in range(dimensions):
                mutated[d] = np.clip(mutated[d], bounds[d][0], bounds[d][1])

            # Crossover
            trial = np.copy(pop[i])

            # Ensure that at least one variable changes randomly
            j_rand = np.random.randint(0, dimensions)

            for j in range(dimensions):
                if np.random.rand() < CR or j == j_rand:
                    trial[j] = mutated[j]

            # Greedy Selection
            trial_fitness = objective_func(trial)
            if trial_fitness < fitness[i]:
                pop[i] = trial
                fitness[i] = trial_fitness

                # Update the global best result
                if trial_fitness < best_score:
                    best_score = trial_fitness
                    best_vector = trial

        print(f"Generation {generation}: Best Fitness = {best_score:.6f}")

    return best_vector, best_score, generation_reached_target

### Experiment Configuration
Below are the key parameters defined for the optimization test:

*   **Search Space Definition (Bounds):** The `problem_bounds` variable defines the two-dimensional search space for the $x$ and $y$ variables. The interval $[-32.768, 32.768]$ represents the **literature standard (benchmark)** for the Ackley function. Such a wide range drastically increases the difficulty of the problem by creating thousands of local minima, allowing for a reliable evaluation of the exploration capabilities of the tested algorithms.
*   **Target Precision (`target_val = 0.01`):** This acts as the convergence threshold. The experiment tracks exactly at which generation the algorithm's best solution reaches a value within $0.01$ of the true global minimum ($0.0$).
*   **Tested Population Sizes (`population_sizes = [10, 50, 100]`):** A comparative analysis of these three variants will reveal how the number of individuals influences the balance between global exploration (finding the right valley) and local exploitation (fine-tuning the solution).


In [16]:
# Standard, larger literature bounds for the Ackley function benchmark
problem_bounds = [(-32.768, 32.768), (-32.768, 32.768)]

# Convergence threshold (distance to the true minimum)
target_val = 0.01

# Population sizes selected for comparative analysis
population_sizes = [10, 50, 100]

In [18]:
print(
    f"Running Differential Evolution Experiments (Bounds: {problem_bounds[0]})...\n"
)

for pop in population_sizes:
    print(f"{'='*15} EXPERIMENT: pop_size = {pop} {'='*15}")
    best_sol, best_val, target_gen = differential_evolution_experiment(
        ackley,
        problem_bounds,
        pop_size=pop,
        F=0.6,
        CR=0.8,
        max_iter=50,
        target_quality=target_val,
    )

    print(f"Final best point (x, y): {best_sol}")
    print(f"Function value at this point: {best_val:.10f}")

    if target_gen is not None:
        print(
            f"SUCCESS: Reached target quality ({target_val}) at generation: {target_gen}"
        )
    else:
        print(
            f"FAILURE: Did not reach target quality ({target_val}) within 300 generations."
        )
    print("\n")


Running Differential Evolution Experiments (Bounds: (-32.768, 32.768))...

=============== EXPERIMENT: pop_size = 10 ===============
Generation 0: Best Fitness = 14.727858
Generation 1: Best Fitness = 13.639279
Generation 2: Best Fitness = 13.639279
Generation 3: Best Fitness = 12.879944
Generation 4: Best Fitness = 7.080327
Generation 5: Best Fitness = 7.080327
Generation 6: Best Fitness = 3.798045
Generation 7: Best Fitness = 3.798045
Generation 8: Best Fitness = 3.322111
Generation 9: Best Fitness = 0.405049
Generation 10: Best Fitness = 0.405049
Generation 11: Best Fitness = 0.405049
Generation 12: Best Fitness = 0.334562
Generation 13: Best Fitness = 0.334562
Generation 14: Best Fitness = 0.250248
Generation 15: Best Fitness = 0.250248
Generation 16: Best Fitness = 0.215771
Generation 17: Best Fitness = 0.105412
Generation 18: Best Fitness = 0.080237
Generation 19: Best Fitness = 0.051876
Generation 20: Best Fitness = 0.051876
Generation 21: Best Fitness = 0.011584
Generation 22: 

### Analysis of Results and Discussion

#### 1. Comparison of Convergence Speed (Efficiency)
Based on the experiments with `max_iter = 50`, we recorded the following generational speed to reach the target quality ($0.01$):
*   **`pop_size = 10`**: Reached the target at generation **26**.
*   **`pop_size = 50`**: Reached the target at generation **33**.
*   **`pop_size = 100`**: Reached the target at generation **25**.

The results present a non-linear relationship, which perfectly demonstrates the stochastic nature of evolutionary computations and the phenomenon of random exploration.

#### 2. Impact on Exploration (Global Search)
*   **Small Population (`pop_size = 10`):** Its high speed (26 generations) is a textbook example of **lucky initialization**. With only 10 individuals, the initial random seed placed an individual on a favorable path near the central basin. Because the population is small, it collapsed toward this point instantly. However, this is statistically unstable; in a batch of multiple independent runs, a size of 10 heavily suffers from premature convergence in Ackley's local traps.
*   **Medium Population (`pop_size = 50`):** This variant took the longest (33 generations) because it experienced a temporary stagnation. As seen in the logs between generations 2 and 8, the best fitness stalled at `2.690940`. The population was divided exploring different promising sub-regions (high exploration) and needed a few generations to coordinate the vectors toward the absolute center.
*   **Large Population (`pop_size = 100`):** Showed the most robust and stable performance (25 generations). Due to the massive number of individuals, the initial population achieved superior coverage, starting with an impressive initial fitness of `5.079660`. It did not experience any stagnation phases, moving continuously and reliably to the target.

#### 3. Impact on Exploitation (Local Fine-Tuning)
*   **`pop_size = 10`**: Reached the threshold fast but lacked long-term genetic variety to compress further efficiently.
*   **`pop_size = 100`**: Maintained a vast pool of diverse differential vectors, allowing for a highly precise and smooth local dive, stabilizing near zero with high floating-point confidence.

#### Conclusion
A small population can occasionally outperform a medium one due to pure random luck during initialization. However, only a large population (`pop_size = 100`) guarantees steady, reliable exploration of the search space, preventing stagnation and ensuring high-precision exploitation.


## Simulated Annealing algorithm

### Experiment Configuration: Simulated Annealing (SA)
Below are the key parameters and variables defined for the Simulated Annealing optimization test:

*   **Search Space Definition (Bounds):** The `problem_bounds` variable maintains the same two-dimensional search space for the $x$ and $y$ variables, bounded within $[-32.768, 32.768]$. This allows for a fair and direct comparison with the Differential Evolution algorithm under identical literature benchmark difficulties.
*   **Target Precision (`target_val = 0.01`):** This serves as the identical convergence threshold. The experiment tracks the exact internal iteration (step) at which the single trajectory of Simulated Annealing drops below $0.01$, allowing us to evaluate and compare the convergence speed of both paradigms.
*   **Trajectory-Based Variables (`step_counter` & `reached_at_step`):** Unlike the population-based approach, Simulated Annealing tracks a single solution across the search space. These global counters dynamically monitor the internal step of the algorithm to log the precise moment of success without the need for complex object-oriented structures.

In [10]:
# Standard, larger literature bounds for the Ackley function benchmark
problem_bounds = [(-32.768, 32.768), (-32.768, 32.768)]

# Convergence threshold (distance to the true minimum)
target_val = 0.01

# Tested temperature variants for Simulated Annealing (replacing population sizes)
temperature_variants = [10, 50, 100]

In [1]:
!uv pip install scipy

Resolved 2 packages in 542ms                                         
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/14.01 MiB           
⠙ Preparing packages... (0/1)------------------- 16.00 KiB/14.01 MiB         
⠙ Preparing packages... (0/1)------------------- 32.00 KiB/14.01 MiB         
⠙ Preparing packages... (0/1)------------------- 48.00 KiB/14.01 MiB         
⠙ Preparing packages... (0/1)------------------- 62.70 KiB/14.01 MiB         
⠙ Preparing packages... (0/1)------------------- 78.70 KiB/14.01 MiB         
⠙ Preparing packages... (0/1)------------------- 94.70 KiB/14.01 MiB         
⠙ Preparing packages... (0/1)------------------- 110.70 KiB/14.01 MiB        
⠙ Preparing packages... (0/1)------------------- 126.70 KiB/14.01 MiB        
⠙ Preparing packages... (0/1)------------------- 142.70 KiB/14.01 MiB        
⠙ Preparing packages... (0/1)------------------- 158.70 KiB/14.01 MiB

In [11]:
from scipy.optimize import dual_annealing


def simulated_annealing_callback(x, f, context):
    global step_counter, reached_at_step
    step_counter += 1

    # Check if the current solution value drops below our target precision
    if f <= target_val and reached_at_step is None:
        reached_at_step = step_counter

In [12]:
print(f"Running Simulated Annealing Experiments (Bounds: {problem_bounds})...\n")

for temp in temperature_variants:
    print(f"{'='*15} EXPERIMENT: initial_temp = {temp} {'='*15}")

    step_counter = 0
    reached_at_step = None

    # Execution of the optimization with maxiter limited to 50
    result_sa = dual_annealing(
        ackley,
        bounds=problem_bounds,
        maxiter=50,
        initial_temp=temp,
        callback=simulated_annealing_callback,
        seed=42
    )

    print(f"Final best point (x, y): {result_sa.x}")
    print(f"Function value at this point: {result_sa.fun:.10f}")

    if reached_at_step is not None:
        print(f"SUCCESS: Reached target quality ({target_val}) at step: {reached_at_step}")
    else:
        print(f"FAILURE: Did not reach target quality ({target_val}) within 50 iterations.")
    print("\n")


Running Simulated Annealing Experiments (Bounds: [(-32.768, 32.768), (-32.768, 32.768)])...

=============== EXPERIMENT: initial_temp = 10 ===============
Final best point (x, y): [-2.34798566e-09 -1.12479325e-10]
Function value at this point: 0.0000000066
SUCCESS: Reached target quality (0.01) at step: 19


=============== EXPERIMENT: initial_temp = 50 ===============
Final best point (x, y): [-3.25031133e-08  9.52166549e-01]
Function value at this point: 2.5799275570
FAILURE: Did not reach target quality (0.01) within 50 iterations.


=============== EXPERIMENT: initial_temp = 100 ===============
Final best point (x, y): [-4.54188339e-09 -1.47413343e-10]
Function value at this point: 0.0000000129
SUCCESS: Reached target quality (0.01) at step: 16




### Analysis of Simulated Annealing (SA) Results

#### 1. Evaluation of Temperature Variants
By limiting the optimization process to just 50 cooling iterations (`max_iter = 50`), we exposed a fascinating, non-linear relationship between the initial system temperature (`initial_temp`) and the algorithm's capability to find the global minimum:
*   **`initial_temp = 10`**: Achieved **SUCCESS** at step **19**.
*   **`initial_temp = 50`**: Ended in **FAILURE** (stuck at a fitness value of `2.579927`).
*   **`initial_temp = 100`**: Achieved **SUCCESS** at step **16**.

#### 2. Why did `initial_temp = 50` fail? (The Danger of Stagnation)
This failure perfectly illustrates the classic danger of poorly tuned temperature in trajectory-based heuristics:
*   At `initial_temp = 50`, the algorithm had enough thermal energy to jump out of the very small outer ripples, but not enough energy to bounce out of a medium-sized local trough before the 50-iteration limit cut it off.
*   As the temperature began its geometric decay, the step size shrank, and the algorithm got permanently trapped at the coordinates `(0.0, 0.95)`, which represents one of the strongest deceptive local minima surrounding the central Ackley well.

#### 3. Why did `initial_temp = 10` and `100` both succeed?
*   **The Low Temperature Case (`initial_temp = 10`):** With a low initial temperature, the algorithm behaves more like a greedy local hill-climber (or gradient descent). It did not waste iterations making massive, chaotic jumps across the landscape. By a stroke of statistical luck in the initial random placement, it started near an orientation where it could immediately roll straight down into the global basin, reaching the target by step 19.
*   **The High Temperature Case (`initial_temp = 100`):** This is the ideal behavior of Simulated Annealing. The massive initial temperature allowed the single point to perform radical, highly volatile jumps across the $[-32.768, 32.768]$ grid. This high energy successfully forced the algorithm to bypass the deceptive traps, dropping it straight into the central crater almost immediately, logging a success at step 16.

#### Final Synthesis: DE vs. SA under 50-Iteration Restrictions
When resources are heavily constrained (only 50 iterations allowed), **Differential Evolution (DE)** proves to be a more reliable choice because its multi-point population guarantees steady convergence regardless of population settings. **Simulated Annealing (SA)** can be incredibly fast (0.1 seconds) and effective, but it is highly volatile. A wrong hyperparameter choice (like `initial_temp = 50`) can cause a single-point trajectory to freeze inside a local trap, resulting in an optimization failure.